# Axiom as a quality-diversity search — measuring archive coverage and QD-score

This notebook runs **the real MAP-Elites search in this repository** (`axiom::run_evolution`, with
`axiom::fitness::evaluate` doing the actual rollouts) on Colab, and measures what the search
actually produces: how much of the behaviour archive gets filled, how the QD-score grows, and how
much of either survives changing the seed.

Nothing here is re-implemented in Python. The notebook shells out to a headless harness
(`examples/qd_sweep.rs`) that calls the engine and prints one JSON document; every number below is
parsed out of that.

### What this notebook establishes

1. That the engine builds and runs headlessly on a Colab runtime.
2. **Coverage and QD-score trajectories** across several seeds, reported as mean and spread rather
   than one flattering run.
3. **How much of the result is reproducible and how much is seed luck** — including whether an early
   ranking of seeds predicts the late one (it does not).
4. **Where the archive's coverage ceiling actually is**, which turns out to be a property of the
   mutation operator rather than of the search budget.
5. **Determinism**: the same seed reproduces a byte-identical report, checked both in-process and
   across two separate process invocations.

> No GPU. Set the Colab runtime to **CPU** — this is a single-threaded Rust simulation and a GPU
> would sit idle and still bill you.

## Working loop: this notebook lives in git, and runs on Colab

Open it straight from GitHub — Colab reads the notebook out of the repository:

```
https://colab.research.google.com/github/charlieijk/Axiom/blob/main/notebooks/01_quality_diversity.ipynb
```

The repository is **private**, so the first open prompts you to authorize Colab's GitHub integration
with private-repo access. That same grant is what lets Colab save back.

**Saving your changes:** `File -> Save a copy in GitHub`, targeting a **branch, never `main`**, then
open a PR as usual.

**Before you save, clear the outputs** (`Edit -> Clear all outputs`). Committed outputs make review
diffs unreadable and quietly turn a stale run into apparent evidence. Results worth keeping go to
`notebooks/results/` with a provenance stamp attached, not into cell output.

## 1. Runtime setup

The engine is Rust, edition 2024, MSRV **1.85**. Colab has no Rust toolchain, so the cell below
installs one with `rustup` and clones the repository.

**Be honest about the wait.** On a cold Colab runtime this cell takes **several minutes**, almost
all of it downloading and unpacking the toolchain. The crate itself is cheap: `axiom` depends only
on `serde` and `serde_json`, and a cold release build of the harness — from an empty `target/`,
timed — took **17.7 seconds** on an M-series laptop. If the cell seems stuck, it is rustup, not the
build.

**On the token:** a private clone needs credentials. Store a GitHub PAT in **Colab Secrets** (the
key icon in the left sidebar) as `GITHUB_TOKEN` — never paste it into a cell you will commit. After
cloning, the cell supplies it through a temporary `GIT_ASKPASS` helper. The clone URL, process
arguments, command output, errors, and stored git remote all remain token-free.

In [ ]:
import os, subprocess, sys, pathlib, tempfile, time

IN_COLAB = "google.colab" in sys.modules
REPO   = "charlieijk/Axiom"
BRANCH = "main"

def sh(cmd, **kw):
    '''Run a shell command, failing loudly instead of silently continuing.'''
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    if r.stdout.strip(): print(r.stdout.strip()[-2000:])
    if r.returncode != 0:
        print(r.stderr.strip()[-2000:], file=sys.stderr)
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r.stdout

if IN_COLAB:
    started = time.time()
    # Minimal profile: no docs, no rust-analyzer. Everything this notebook needs
    # is cargo + rustc, and the download is the expensive part.
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs "
       "| sh -s -- -y --profile minimal --default-toolchain stable --no-modify-path")
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
    print(f"toolchain installed in {time.time() - started:.0f}s")

    REPO_DIR = pathlib.Path("/content/Axiom")
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            from getpass import getpass
            token = getpass("GitHub token (input hidden): ")
        if not token:
            raise RuntimeError("GITHUB_TOKEN is required to clone the private repository")

        # Keep the credential out of argv, cell output, errors, and .git/config.
        with tempfile.TemporaryDirectory(prefix="axiom-askpass-") as auth_dir:
            askpass = pathlib.Path(auth_dir) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\n"
                "import os, sys\n"
                "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
                "print('x-access-token' if 'username' in prompt else os.environ['AXIOM_GITHUB_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            clone_env = os.environ.copy()
            clone_env.update({
                "GIT_ASKPASS": str(askpass),
                "GIT_TERMINAL_PROMPT": "0",
                "AXIOM_GITHUB_TOKEN": token,
            })
            try:
                clone = subprocess.run(
                    ["git", "clone", "--depth", "1", "--branch", BRANCH,
                     f"https://github.com/{REPO}.git", str(REPO_DIR)],
                    text=True, capture_output=True, env=clone_env, check=False,
                )
            finally:
                clone_env.pop("AXIOM_GITHUB_TOKEN", None)
                del clone_env
                token = None
        if clone.returncode != 0:
            raise RuntimeError("git clone failed; check the token scope and repository access")
    assert "x-access-token" not in (REPO_DIR / ".git" / "config").read_text(), \
        "the token is still in .git/config — do not push from this runtime"
else:
    # Running locally from inside the checkout — notebooks/ or the repository root.
    here = pathlib.Path.cwd()
    REPO_DIR = here if (here / "Cargo.toml").exists() else here.parent
    if not (REPO_DIR / "Cargo.toml").exists():
        raise RuntimeError(f"could not locate the repository root from {here}")

os.chdir(REPO_DIR)
print("runtime:", "colab" if IN_COLAB else "local")
print("repo:   ", REPO_DIR)

### Verify the toolchain, then build the harness

An MSRV violation shows up as a confusing parse error deep in a dependency. Fail here instead, with
a message that says what is wrong.

In [ ]:
import re

RUSTC_VERSION = subprocess.run("rustc --version", shell=True, text=True,
                               capture_output=True).stdout.strip()
print(RUSTC_VERSION)
m = re.search(r"rustc (\d+)\.(\d+)", RUSTC_VERSION)
if not m:
    raise RuntimeError("could not parse `rustc --version` — is Rust on PATH?")
major, minor = int(m.group(1)), int(m.group(2))
assert (major, minor) >= (1, 85), (
    f"Axiom is edition 2024 with MSRV 1.85; found {major}.{minor}. "
    "Re-run the setup cell, or `rustup update stable`."
)
print("MSRV satisfied.")

In [ ]:
started = time.time()
sh("cargo build --release -p axiom --example qd_sweep")
BUILD_SECONDS = time.time() - started
HARNESS = REPO_DIR / "target" / "release" / "examples" / "qd_sweep"
assert HARNESS.exists(), f"harness not where expected: {HARNESS}"
print(f"harness built in {BUILD_SECONDS:.1f}s -> {HARNESS}")

## 2. Driving the real engine

`examples/qd_sweep.rs` is a thin harness: it calls `axiom::run_evolution` with an explicit seed and
prints one JSON document. It does not contain a copy of the search, the fitness function, or the
archive — those all live in `src/`, and the harness only reads what the engine returns.

Two details worth knowing before reading any number below.

**The QD-score trajectory is measured with prefix runs.** `EvolutionReport` records coverage per
generation but keeps only the *final* archive, so the per-generation QD-score cannot be read out of
a single report. Adding a field to `EvolutionReport` would change its serialization — and the
checked-in Arm64 evidence pins a digest over exactly that serialization — so the harness instead
re-runs the same seed with `generations = 1..=N` and reads the archive at each length. That is exact
rather than approximate: the generation loop skips reproduction on its final iteration, and
reproduction is the only step drawing RNG after the archive insert, so a run of length *k* reaches
the same state as the first *k* generations of a longer run. **The harness does not assume this — it
checks it** and reports `prefix_consistent`, which is asserted below. The cost is the O(N²) shape of
the sweep, which is affordable only because a full run takes well under a second.

**QD-score here can go down.** The textbook QD-score is the sum of elite fitness over occupied
cells, and it assumes fitness is non-negative. Axiom's fitness subtracts a tumble penalty without a
floor, so discovering a *new* niche occupied by a tumbling creature can lower the raw sum. The
harness reports both the raw `qd_score` and `qd_score_positive` (each cell floored at zero); this
notebook reads the raw one and reports how many elites are negative.

In [ ]:
import json

def run_sweep(seeds, generations, population=32, steps=240, trajectory=True):
    '''Invoke the harness and return its JSON document.'''
    args = [str(HARNESS),
            "--seeds", ",".join(str(s) for s in seeds),
            "--generations", str(generations),
            "--population", str(population),
            "--steps", str(steps)]
    if not trajectory:
        args.append("--no-trajectory")
    print("$", " ".join(args))
    r = subprocess.run(args, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stderr[-2000:], file=sys.stderr)
        raise RuntimeError(f"harness failed ({r.returncode})")
    document = json.loads(r.stdout)
    assert document["schema_version"] == 1, "harness output schema changed"
    return document

probe = run_sweep([1], generations=3, population=8, steps=40)
print(json.dumps(probe["behaviour_space"], indent=2))
print("engine:", probe["engine"])

### The provenance stamp

Every result this notebook saves carries the commit it ran against, the toolchain, and the time. A
metric without this is a claim; with it, it is a measurement someone else can re-run.

In [ ]:
from datetime import datetime, timezone

def git(*args):
    return subprocess.run(["git", *args], check=True, text=True, capture_output=True,
                          cwd=REPO_DIR).stdout.strip()

def provenance():
    return {
        "commit": git("rev-parse", "HEAD"),
        "tree": git("rev-parse", "HEAD^{tree}"),
        "dirty_worktree": bool(git("status", "--porcelain")),
        "engine": "axiom::run_evolution (MAP-Elites) via examples/qd_sweep.rs",
        "rustc": RUSTC_VERSION,
        "build_seconds": round(BUILD_SECONDS, 1),
        "verified_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "runtime": "colab" if IN_COLAB else "local",
    }

PROV = provenance()
if PROV["dirty_worktree"]:
    print("NOTE: worktree is dirty — these numbers do not correspond to a clean commit.")
PROV

## 3. The measurement: coverage and QD-score over generations

The configuration below is the engine's own default population and rollout length, run for 40
generations across **8 seeds**. The archive is 12x8 = 96 cells over (stable distance, body count).

One seed is an anecdote. Everything from here on is reported as mean and spread.

In [ ]:
SEEDS       = [1, 2, 3, 4, 5, 6, 7, 8]
GENERATIONS = 40
POPULATION  = 32
STEPS       = 240

main = run_sweep(SEEDS, generations=GENERATIONS, population=POPULATION, steps=STEPS)

CELLS = main["config"]["archive_cells"]
assert all(s["prefix_consistent"] for s in main["seeds"]), (
    "a prefix run diverged from the full run — the trajectory method is invalid, "
    "and every per-generation number below would be fiction"
)
print(f"prefix-consistency check passed for all {len(SEEDS)} seeds")
print(f"archive: {main['config']['archive_width']}x{main['config']['archive_height']}"
      f" = {CELLS} cells, {main['seeds'][0]['final']['evaluated_genomes']} genomes evaluated per seed")
print(f"harness wall clock: {main['wall_clock_seconds']:.1f}s")

In [ ]:
import statistics as st

def column(document, key):
    return [s["final"][key] for s in document["seeds"]]

print(f"{'seed':>5} {'cells':>6} {'coverage':>9} {'QD-score':>10} {'best fitness':>13} {'neg elites':>11}")
for s in main["seeds"]:
    f = s["final"]
    print(f"{s['seed']:>5} {f['occupied_cells']:>6} {f['coverage']*100:>8.1f}% "
          f"{f['qd_score']:>10.0f} {f['best_fitness']:>13.1f} {f['negative_fitness_elites']:>11}")

def spread(values, label, scale=1.0, fmt="{:.1f}"):
    mean, sd = st.mean(values) * scale, st.stdev(values) * scale
    lo, hi = min(values) * scale, max(values) * scale
    print(f"{label:<14} mean {fmt.format(mean)}  sd {fmt.format(sd)}  "
          f"range [{fmt.format(lo)}, {fmt.format(hi)}]  cv {sd/mean:.2f}")

print()
spread(column(main, "coverage"), "coverage %", 100)
spread(column(main, "qd_score"), "QD-score")
spread(column(main, "best_fitness"), "best fitness")

In [ ]:
import matplotlib.pyplot as plt

def band(ax, document, key, colour, label, scale=1.0):
    gens = [p["generation"] for p in document["seeds"][0]["trajectory"]]
    series = [[p[key] * scale for p in s["trajectory"]] for s in document["seeds"]]
    means = [st.mean(v) for v in zip(*series)]
    sds   = [st.stdev(v) for v in zip(*series)]
    for run in series:
        ax.plot(gens, run, color=colour, alpha=0.18, linewidth=1)
    ax.plot(gens, means, color=colour, linewidth=2.2, label=f"{label} (mean of {len(series)} seeds)")
    ax.fill_between(gens,
                    [m - s for m, s in zip(means, sds)],
                    [m + s for m, s in zip(means, sds)],
                    color=colour, alpha=0.18, label="+/- 1 sd")
    ax.set_xlabel("generation"); ax.grid(alpha=0.3); ax.legend(loc="upper left", fontsize=8)
    return means

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))
cov_means = band(left,  main, "coverage", "tab:blue",  "archive coverage", 100)
qd_means  = band(right, main, "qd_score", "tab:orange", "QD-score")
left.set_ylabel("archive coverage (% of 96 cells)")
left.set_title("Coverage over generations")
right.set_ylabel("QD-score (sum of elite fitness)")
right.set_title("QD-score over generations")
plt.tight_layout(); plt.show()

print(f"coverage: {cov_means[0]:.1f}% at gen 1 -> {cov_means[-1]:.1f}% at gen {GENERATIONS}")
print(f"QD-score: {qd_means[0]:.0f} at gen 1 -> {qd_means[-1]:.0f} at gen {GENERATIONS}")

**Read the faint lines, not just the bold one.** The individual seeds fan out steadily; the mean is
a summary of a wide distribution, not a description of a typical run. Note also that coverage is a
staircase with long flat treads — whole stretches of generations add no new niche at all — while the
QD-score keeps creeping up over those same stretches, because the search is still improving elites
inside cells it has already found.

## 4. How much of this is reproducible, and how much is seed luck?

The spread above is large enough that a single run would be a poor description of the method. This
section asks the sharper question: **does an early result predict a late one?** If it does not, then
comparing two variants of the search on a short run — a very common way to "tune" a QD algorithm —
is measuring noise.

Spearman rank correlation over the 8 seeds, early ranking versus final ranking.

In [ ]:
def spearman(a, b):
    def rank(values):
        order = sorted(range(len(values)), key=lambda i: values[i])
        ranks = [0] * len(values)
        for position, index in enumerate(order):
            ranks[index] = position + 1
        return ranks
    ra, rb, n = rank(a), rank(b), len(a)
    return 1 - 6 * sum((ra[i] - rb[i]) ** 2 for i in range(n)) / (n * (n * n - 1))

def at(generation, key):
    return [s["trajectory"][generation - 1][key] for s in main["seeds"]]

final_qd, final_best = at(GENERATIONS, "qd_score"), at(GENERATIONS, "best_fitness")
RANK_STABILITY = {}
for early in (5, 10, 20, 30):
    RANK_STABILITY[early] = {
        "qd_score": spearman(at(early, "qd_score"), final_qd),
        "best_fitness": spearman(at(early, "best_fitness"), final_best),
    }
    print(f"gen {early:>2} vs gen {GENERATIONS}:  "
          f"QD-score rho = {RANK_STABILITY[early]['qd_score']:+.2f}   "
          f"best-fitness rho = {RANK_STABILITY[early]['best_fitness']:+.2f}")

order_early = [s["seed"] for s in sorted(main["seeds"],
               key=lambda s: -s["trajectory"][9]["qd_score"])]
order_final = [s["seed"] for s in sorted(main["seeds"],
               key=lambda s: -s["final"]["qd_score"])]
print(f"\nseeds ranked by QD-score at gen 10:  {order_early}")
print(f"seeds ranked by QD-score at gen {GENERATIONS}:  {order_final}")

**This is the finding, and it is not a flattering one.** Through generation 10 the ranking of seeds
is essentially uncorrelated with the ranking at generation 40 — a coin flip dressed up as a
measurement, and on best fitness the early rank correlation is briefly *negative*. A partial signal
appears by generation 20, and only by generation 30 does the ordering firm up. So a short pilot run
in this sandbox tells you which seed got lucky early, not which configuration is better; any
comparison between two search variants needs several seeds *and* a budget long enough for the
ranking to have stabilised. The sample is 8 seeds, so these rho values carry wide uncertainty
themselves — the safe reading is the shape (no signal early, signal late), not the exact numbers.

## 5. Does coverage plateau — and if so, against what?

The obvious next question is whether more generations keep paying. Re-run at four budgets, without
the trajectory sweep so the cost stays linear.

In [ ]:
BUDGETS = [20, 40, 80, 160]
budget_runs = {g: run_sweep(SEEDS, generations=g, population=POPULATION,
                            steps=STEPS, trajectory=False) for g in BUDGETS}

print(f"{'gens':>5} {'evals':>7} {'coverage %':>18} {'QD-score':>18} {'best fitness':>17}")
BUDGET_TABLE = {}
for g, document in budget_runs.items():
    cov  = [v * 100 for v in column(document, "coverage")]
    qd   = column(document, "qd_score")
    best = column(document, "best_fitness")
    BUDGET_TABLE[g] = {
        "evaluated_genomes": document["seeds"][0]["final"]["evaluated_genomes"],
        "coverage_mean": st.mean(cov), "coverage_sd": st.stdev(cov),
        "qd_mean": st.mean(qd), "qd_sd": st.stdev(qd),
        "best_mean": st.mean(best), "best_sd": st.stdev(best),
    }
    row = BUDGET_TABLE[g]
    print(f"{g:>5} {row['evaluated_genomes']:>7} "
          f"{row['coverage_mean']:>10.1f} +/-{row['coverage_sd']:>5.1f} "
          f"{row['qd_mean']:>11.0f} +/-{row['qd_sd']:>5.0f} "
          f"{row['best_mean']:>10.1f} +/-{row['best_sd']:>5.1f}")

Coverage keeps climbing across the whole range, so on the face of it nothing has plateaued. But
"coverage" is a percentage of the *grid*, and the grid is not the same thing as the space the search
can actually reach. Check that before concluding anything.

In [ ]:
axes = main["behaviour_space"]
elites = [e for s in budget_runs[160]["seeds"] for e in s["final"]["elites"]]
body_counts = sorted({e["body_count"] for e in elites})
rows_used   = sorted({e["cell"][1] for e in elites})
cols_used   = sorted({e["cell"][0] for e in elites})

WIDTH  = main["config"]["archive_width"]
HEIGHT = main["config"]["archive_height"]
REACHABLE = WIDTH * len(rows_used)
CEILING = 100 * REACHABLE / CELLS

print(f"y axis '{axes['y_axis']['label']}' spans "
      f"[{axes['y_axis']['min']}, {axes['y_axis']['max']}) over {HEIGHT} bins")
print(f"body counts the search ever produced: {[int(b) for b in body_counts]}")
print(f"archive rows ever occupied:           {rows_used} of {list(range(HEIGHT))}")
print(f"archive columns ever occupied:        {cols_used} of {list(range(WIDTH))}")
print(f"\nreachable cells: {WIDTH} x {len(rows_used)} = {REACHABLE} of {CELLS}"
      f"  ->  coverage ceiling {CEILING:.1f}%")
print("\nbudget   coverage of grid   coverage of REACHABLE space")
for g, row in BUDGET_TABLE.items():
    print(f"{g:>6}   {row['coverage_mean']:>13.1f}%   {row['coverage_mean']/CEILING*100:>22.1f}%")

In [ ]:
seed_index = max(range(len(main["seeds"])),
                 key=lambda i: main["seeds"][i]["final"]["occupied_cells"])
shown = budget_runs[160]["seeds"][seed_index]
grid = [[float("nan")] * WIDTH for _ in range(HEIGHT)]
for e in shown["final"]["elites"]:
    grid[e["cell"][1]][e["cell"][0]] = e["fitness"]

fig, ax = plt.subplots(figsize=(7.2, 4.4))
image = ax.imshow(grid, origin="lower", cmap="viridis", aspect="auto")
for row in range(HEIGHT):
    if row not in rows_used:
        ax.axhspan(row - 0.5, row + 0.5, color="0.85", zorder=3)
        ax.text(WIDTH / 2 - 0.5, row, "unreachable: no genome has this body count",
                ha="center", va="center", fontsize=7.5, color="0.25", zorder=4)
fig.colorbar(image, ax=ax, label="elite fitness")
ax.set_xlabel(f"{axes['x_axis']['label']} bin "
              f"[{axes['x_axis']['min']:g}, {axes['x_axis']['max']:g})")
ax.set_ylabel(f"{axes['y_axis']['label']} bin "
              f"[{axes['y_axis']['min']:g}, {axes['y_axis']['max']:g})")
ax.set_title(f"MAP-Elites archive, seed {shown['seed']}, {shown['final']['generations']} generations"
             f" — {shown['final']['occupied_cells']}/{CELLS} cells filled")
plt.tight_layout(); plt.show()

**The ceiling is in the mutation operator, not in the search budget.** `BodyGenome::seed_quadruped`
starts every genome at 5 segments and `BodyGenome::mutate` can only *add* segments, capped at 10 —
it never removes one. The body-count axis is declared over `[1, 10)`, so the bottom three rows of
the archive describe creatures the search is structurally incapable of producing. Roughly a third of
the grid can never be filled, and the headline "coverage" figure is deflated by exactly that much.

Measured against the space the search can actually reach, the picture changes: what looked like a
half-explored archive at the largest budget is most of the reachable region. The honest reading of
the budget table is that **the extra compute buys diversity inside a fixed region, while the
champion barely moves** — and the improvement in best fitness across an 8x budget increase is of the
same order as the seed-to-seed standard deviation, which means a single-seed before/after comparison
at these budgets cannot distinguish a real improvement from a lucky draw.

This is a defect worth recording rather than a curiosity: the coverage number reported by
`axiom evolve` is not a percentage of anything achievable. Either the axis range should track what
the operator can produce, or the mutation operator should be able to remove a segment.

## 6. Determinism: the same seed reproduces an identical archive

`src/rng.rs` is a seeded generator and the engine takes the seed from its config, so a run should be
exactly reproducible. "Should be" is not a measurement. Two checks:

1. **In-process** — the harness runs the same config twice and compares the entire
   `EvolutionReport` with `==`: champion, every archive elite, generation history and full lineage.
   Comparing the whole report stops a deterministic-but-wrong result from hiding behind a few
   unchanged headline metrics.
2. **Across processes** — the cell below invokes the harness a second time, in a separate process,
   and compares the FNV-1a digest of the serialized report.

In [ ]:
determinism = main["determinism"]
assert determinism["reports_identical"], "same seed produced different reports in one process"
assert determinism["report_digest_a"] == determinism["report_digest_b"]
print(f"in-process:     seed {determinism['seed']} -> {determinism['report_digest_a']} (twice, reports equal)")

repeat = run_sweep(SEEDS, generations=GENERATIONS, population=POPULATION,
                   steps=STEPS, trajectory=False)
mismatches = [(a["seed"], a["final"]["report_digest"], b["final"]["report_digest"])
              for a, b in zip(main["seeds"], repeat["seeds"])
              if a["final"]["report_digest"] != b["final"]["report_digest"]]
assert not mismatches, f"digest differed across processes: {mismatches}"
print(f"across processes: all {len(SEEDS)} seeds reproduced their report digest exactly")
for s in main["seeds"]:
    print(f"  seed {s['seed']:>2}  {s['final']['report_digest']}")

DETERMINISM = {
    "in_process_reports_identical": determinism["reports_identical"],
    "digest_algorithm": determinism["digest_algorithm"],
    "digests": {s["seed"]: s["final"]["report_digest"] for s in main["seeds"]},
    "cross_process_match": True,
}

## 7. Save the result with its provenance

In [ ]:
target = pathlib.Path(REPO_DIR) / "notebooks" / "results" / "quality_diversity.json"
target.parent.mkdir(parents=True, exist_ok=True)

def summarise(document):
    return {
        "generations": document["config"]["generations"],
        "evaluated_genomes_per_seed": document["seeds"][0]["final"]["evaluated_genomes"],
        "per_seed": {s["seed"]: {"occupied_cells": s["final"]["occupied_cells"],
                                 "coverage": s["final"]["coverage"],
                                 "qd_score": s["final"]["qd_score"],
                                 "best_fitness": s["final"]["best_fitness"],
                                 "report_digest": s["final"]["report_digest"]}
                     for s in document["seeds"]},
    }

payload = {
    "provenance": PROV,
    "config": {"seeds": SEEDS, "generations": GENERATIONS, "population": POPULATION,
               "evaluation_steps": STEPS, "task": main["config"]["task"],
               "archive": f"{WIDTH}x{HEIGHT}", "archive_cells": CELLS},
    "behaviour_space": axes,
    "main_run": summarise(main),
    "trajectory_mean": {"coverage_percent": cov_means, "qd_score": qd_means},
    "budget_sweep": BUDGET_TABLE,
    "rank_stability_spearman": RANK_STABILITY,
    "reachability": {"rows_occupied": rows_used, "body_counts_observed": body_counts,
                     "reachable_cells": REACHABLE, "coverage_ceiling_percent": CEILING},
    "determinism": DETERMINISM,
}
target.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")
print(f"wrote {target}")
print(json.dumps(payload["main_run"]["per_seed"], indent=2)[:600], "...")

## Limits of what this notebook shows

Stated plainly, so nothing downstream quietly over-claims on top of it.

- **What "improvement" means here is whatever `axiom::fitness::evaluate` rewards**, and that is a
  hand-written weighted sum: stability-discounted forward distance (weighted heaviest), raw forward
  distance, a posture term over uprightness and terminal tilt, a small hop bonus, minus a quadratic
  tumble penalty, minus energy spent, minus a small per-segment cost. A rising QD-score means the
  archive holds creatures that score better on *that formula* — not that they walk better in any
  sense a person would recognise. Changing a coefficient would change every number above.
- **Eight seeds is a small sample.** The standard deviations quoted are estimated from eight points
  and are themselves noisy; treat them as an order of magnitude for the spread, not a precise
  interval. Nothing here is a significance test.
- **One task, one archive geometry.** Everything is `RoughTerrain` on a 12x8 (stable distance, body
  count) grid. The reachability ceiling in section 5 is a property of *that* axis choice; a
  different behaviour space would move it.
- **QD-score is not comparable across configurations.** It is an unnormalised sum over occupied
  cells, so it grows with both the number of cells and their fitness, and it is sensitive to the
  archive resolution. Only compare it between runs that share a grid and a fitness function.
- **The trajectory uses prefix runs, not a live per-generation readout.** The notebook asserts
  prefix-consistency rather than assuming it, but if that assertion ever fails the trajectory panels
  are invalid — the assertion is load-bearing, not decorative.
- **The Arm64 allocation figures are not measured here.** The documented "94.38% fewer allocations"
  is a *stored* benchmark result in `docs/benchmarks/`, recorded on Apple Silicon behind the
  `allocation-counting` feature and pinned by `tests/arm64_evidence.rs`. This notebook neither runs
  that benchmark nor reproduces it, and a Colab runtime is x86-64 anyway, so it could not produce an
  authoritative Arm64 figure even if it did — the benchmark stamps `authoritative_arm64: false` off
  Arm hardware for exactly that reason. Quote that number as a recorded measurement on named
  hardware, never as something this notebook confirmed.
- **Nothing is trained.** MAP-Elites here is mutation-only search over genomes; there is no gradient,
  no learned model, and no held-out evaluation. "Elite" means best-so-far in a cell, on the same
  deterministic rollout it was selected by, so there is no generalisation claim available at all.

## What cannot run on Colab

- **The Arm64 allocation benchmark**, as above: Colab runtimes are x86-64, so
  `examples/arm64_optimization_benchmark.rs` will refuse to claim authority there.
- **The GUI** (`cargo run -- gui`) serves on localhost; a Colab runtime cannot show it to you
  without tunnelling, and the replay browser is the interesting part of it.
- **The terminal animation** (`axiom animate`) needs a real TTY.
- **`crates/axiom-field`** pulls in `rapier3d` and is a considerably heavier build than the
  `-p axiom --example qd_sweep` used here; it is deliberately out of scope for this notebook.

## Next

The reachability defect in section 5 is the actionable one: `axiom evolve` reports a coverage
percentage against a grid a third of which no genome can occupy. Fixing it means either narrowing
`Axis::BodyCount`'s declared range to what the operator produces, or giving `BodyGenome::mutate` a
segment-removal step — and the second is the more interesting experiment, because it would let the
search explore a region of behaviour space it has never been able to enter.